# CorrosionAI grain classifier

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Tian-haoyan/CorrosionAI/blob/main/colab/grain_classifier_demo.ipynb)

This notebook runs CorrosionAI inference on uploaded amphibole grain photomicrographs.

Recommended input images:

- one clearly visible amphibole grain per image;
- square, single-grain images, preferably 576 ? 576 pixels;
- RGB optical photomicrographs;
- limited background and limited overlap with other grains;
- no labels, arrows, scale bars, or text covering the grain.

Supported formats: png, jpg, jpeg, bmp, tif, tiff, webp.

## 1. Clone CorrosionAI and install dependencies

In [ ]:
!rm -rf CorrosionAI
!git clone https://github.com/Tian-haoyan/CorrosionAI.git
%cd CorrosionAI
!pip install -r requirements.txt

## 2. Download model checkpoint

The checkpoint is downloaded to `weights/best_acc.pth`. If automatic download fails, check that the Google Drive file is shared as "Anyone with the link can view", or upload the checkpoint manually into the `weights/` folder.

In [ ]:
import os
import gdown

os.makedirs("weights", exist_ok=True)

# Replace this file_id if the checkpoint is moved to a new Google Drive/Release/Zenodo link.
file_id = "1PH_-8IDaCeOTuoSGWGxcdn-xVC30lwof"
url = f"https://drive.google.com/uc?id={file_id}"
checkpoint_path = "weights/best_acc.pth"

gdown.download(url, checkpoint_path, quiet=False)
print("Checkpoint path:", checkpoint_path)
print("Checkpoint exists:", os.path.exists(checkpoint_path))

## 3. Upload images

This cell uploads images into one sample folder named `uploaded_sample`. If you want to analyse several samples separately, run the notebook once per sample or upload images through Google Drive using separate subfolders.

In [ ]:
from google.colab import files
import os

sample_name = "uploaded_sample"
input_root = "external_test_images"
sample_dir = os.path.join(input_root, sample_name)
os.makedirs(sample_dir, exist_ok=True)

uploaded = files.upload()
for filename, content in uploaded.items():
    with open(os.path.join(sample_dir, filename), "wb") as f:
        f.write(content)

print(f"Uploaded {len(uploaded)} files to {sample_dir}")
for fn in sorted(os.listdir(sample_dir))[:20]:
    print(fn)
if len(os.listdir(sample_dir)) > 20:
    print("...")

## 4. Run prediction and calculate CI*

In [ ]:
!python inference/run_external_test.py \
  --input external_test_images \
  --weights weights/best_acc.pth \
  --output outputs/inference_results \
  --device auto

## 5. View prediction outputs

In [ ]:
import pandas as pd
from pathlib import Path

predictions_csv = Path("outputs/inference_results/predictions.csv")
pred = pd.read_csv(predictions_csv)
print("Per-image predictions:")
display(pred.head())

sample_summary_dir = Path("outputs/inference_results/sample_summary/uploaded_sample")
ci_file = sample_summary_dir / "ci_star_summary.txt"
metrics_file = sample_summary_dir / "metrics.txt"

print("\nCI* summary:")
print(ci_file.read_text(encoding="utf-8"))

print("Classification metrics:")
print(metrics_file.read_text(encoding="utf-8"))

## 6. Download all results

In [ ]:
!zip -r corrosionai_results.zip outputs/inference_results
from google.colab import files
files.download("corrosionai_results.zip")